# Statistical Methods in Imaging (SMI) Conference, 2026.
# Empowering Large Language Models with Statistics: A Practical Tutorial for Medical Imaging
**Ernest (Khashayar) Namdar, Dominik A. Deniffel, Pascal Tyrrell**

This tutorial focuses on classifying Acute Ischemic Stroke (AIS) from Radiology reports. We will use the `Label` and `Text` columns for a binary classification task.

Several similar pipelines were discussed in our publication:
```bibtex
@inproceedings{10.1117/12.3084682,
author = {Khashayar Namdar and Saeidehsadat Mirjalili and Lauren Erdman and Dominik A. Deniffel and Keith Brunt and Leo Anthony Celi},
title = {{Comparative evaluation of machine learning and large language model pipelines for identifying acute ischemic stroke in radiology reports}},
volume = {13926},
booktitle = {Medical Imaging 2026: Computer-Aided Diagnosis},
editor = {Axel Wism{\"u}ller and Thomas Martin Deserno},
organization = {International Society for Optics and Photonics},
publisher = {SPIE},
pages = {139261S},
keywords = {Stroke, NLP, Machine Learning, Large Language Models},
year = {2026},
doi = {10.1117/12.3084682},
URL = {https://doi.org/10.1117/12.3084682}
}
```

Also, the source for the dataset is:
```bibtex
@article{10.1371/journal.pone.0212778,
    doi = {10.1371/journal.pone.0212778},
    author = {Kim, Chulho AND Zhu, Vivienne AND Obeid, Jihad AND Lenert, Leslie},
    journal = {PLOS ONE},
    publisher = {Public Library of Science},
    title = {Natural language processing and machine learning algorithm to identify brain MRI reports with acute ischemic stroke},
    year = {2019},
    month = {02},
    volume = {14},
    url = {https://doi.org/10.1371/journal.pone.0212778},
    pages = {1-13},
    number = {2},
}
```


## MediPhi Zero-Shot Generation Demo
In this section, we will demonstrate how to use `microsoft/MediPhi` to answer a clinical question regarding Acute Ischemic Stroke (AIS).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

torch.random.manual_seed(0)

model_name = "microsoft/MediPhi"
# Initialize the Large Language Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda",          # Automatically loads the model weights onto the GPU for faster inference
    torch_dtype="auto",         # Uses the optimal precision (e.g., float16) to save GPU memory
    trust_remote_code=True,     # Required for some newer HuggingFace models with custom architectures
)
# The tokenizer converts raw text into numerical tokens the model can understand
tokenizer = AutoTokenizer.from_pretrained(model_name)

prompt = "Please provide a concise overview of Acute Ischemic Stroke (AIS). Explain what it is, its clinical significance, and what specific findings or signs you would typically look for in diagnostic imaging modalities such as non-contrast CT or MRI to identify it."

messages = [
    {"role": "system", "content": "You are a radiologist."},
    {"role": "user", "content": prompt},
]

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

# Generation hyperparameters
generation_args = {
    "max_new_tokens": 1024,     # Maximum number of tokens the model is allowed to generate
    "return_full_text": False,  # Only return the generated answer, not the prompt
    "do_sample": False,         # Greedy decoding (deterministic). Set to True for creative/random responses
}

output = pipe(messages, **generation_args)
print(output[0]['generated_text'])
